## Preprocesssing 

### Gang's Life Style CSV

In [3]:
# FINAL PREPROCESSING PIPELINE – CLEAN VERSION
# No vendor column, no notes column, no last_scraped_date, no source_file.

import re, os, math
from datetime import datetime
import numpy as np
import pandas as pd

# NLP imports
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

# download NLP resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ---------- USER CONFIG ----------
INPUT_PATH = "/kaggle/input/arniova-studio/Gangs Life Style 14NOV2025.csv"
OUTPUT_PATH = "/kaggle/working/gangslifestyle.csv"
CATEGORY_COVERAGE_THRESHOLD = 0.5
VARIANT_COLOR_COMBINE = True
SUMMARY_SENTENCES = 1
# ------------------------------------

# Helpers
_currency_re = re.compile(r'[^\d.,\-]+')
def parse_price(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() in ['nan','none','null']:
        return np.nan
    s = _currency_re.sub('', s).replace(',', '')
    try: return float(s)
    except:
        nums = re.findall(r'[-+]?\d*\.\d+|\d+', s)
        return float(nums[0]) if nums else np.nan

def normalize_stock(v):
    if pd.isna(v): return 'Unknown'
    s = str(v).strip().lower()
    if s in ['true','yes','1','in stock','available','instock']: return 'In Stock'
    if s in ['false','no','0','out of stock','sold out','not available']: return 'Out of Stock'
    return 'Unknown'

MARKETING_WORDS = {'buy now','best','new','free shipping','hot','sale','discount','offer','trending'}
def clean_title(t):
    if pd.isna(t): return ''
    s = re.sub(r'\s+', ' ', str(t).strip())
    for w in MARKETING_WORDS:
        s = re.sub(r'(?i)\\b' + re.escape(w) + r'\\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()

COLOR_WORDS = {'black','white','red','blue','green','yellow','pink','orange','purple','brown','grey','gray','silver','gold','navy'}
def variant_looks_like_color(v):
    if pd.isna(v) or str(v).strip() == '': return False
    parts = re.split(r'[,/;|-]+', str(v).lower())
    return any(p.strip() in COLOR_WORDS for p in parts)

def clean_functional_text(txt):
    if pd.isna(txt): return ''
    s = re.sub(r'<[^>]+>', ' ', str(txt))
    s = re.sub(r'[\r\n\t]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    words = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(w.lower()) for w in words if w.isalnum())

def extractive_summary(text, n_sentences=1):
    if not text.strip(): return ''
    sents = sent_tokenize(text)
    if len(sents) <= n_sentences: return ' '.join(sents)
    try:
        vec = TfidfVectorizer(stop_words='english')
        X = vec.fit_transform(sents)
        centroid = X.sum(axis=0)
        scores = X.dot(centroid.T).A.ravel()
        idx = scores.argsort()[::-1][:n_sentences]
        idx = sorted(idx)
        return ' '.join(sents[i].strip() for i in idx)
    except:
        return sents[0]

def clean_url(u):
    if pd.isna(u): return ''
    s = str(u).strip()
    return re.sub(r'[\?&]$', '', s)

def looks_like_image_url(u):
    if pd.isna(u): return False
    return bool(re.search(r'\.(jpg|jpeg|png|webp|gif)$', str(u), flags=re.I))

# ---------- Load CSV ----------
df = pd.read_csv(INPUT_PATH, dtype=str)
print("Loaded:", len(df))

# Exact column mapping
col = {
    "title": "Product Name",
    "variant": "Variant Name",
    "sku": "SKU",
    "instock": "In Stock?",
    "price": "Price",
    "original_price": "Original Price",
    "discount_info": "Discount Info",
    "category": "Category",
    "tags": "Tags",
    "functional": "Functional Details",
    "product_url": "Link",
    "image_url": "Main Image URL"
}

working = pd.DataFrame()
for k, v in col.items():
    working[k] = df[v].fillna('') if v in df.columns else ''

# 1. Clean title
working['title'] = working['title'].apply(clean_title)

# 2. Combine variant for colors
if VARIANT_COLOR_COMBINE:
    working['title'] = working.apply(
        lambda r: f"{r['title']} (Color: {r['variant']})" 
        if variant_looks_like_color(r['variant']) else r['title'], axis=1
    )

# 3. SKU fallback
working['sku'] = working['sku'].astype(str).str.strip()
missing = working['sku'] == ''
working.loc[missing, 'sku'] = [f"MISSINGSKU_{i}" for i in range(1, missing.sum()+1)]

# 4. Stock normalize
working['stock_status'] = working['instock'].apply(normalize_stock)

# 5. PRICE + ORIGINAL PRICE FIX (YOU REQUESTED)
working['price_parsed'] = working['price'].apply(parse_price)
working['original_price_parsed'] = working['original_price'].apply(parse_price)

def compute_prices(r):
    p = r['price_parsed']
    o = r['original_price_parsed']

    # If original missing or zero → set original = price
    if pd.isna(o) or o == 0:
        o = p

    # If price missing → set price = original
    if pd.isna(p) and not pd.isna(o):
        p = o

    # Compute discount
    if pd.isna(p) or pd.isna(o):
        disc = np.nan
    else:
        disc = 0 if o == p else round((o - p) / o * 100, 1) if o > p else 0

    return pd.Series([p, o, disc])

working[['price_current', 'price_original', 'discount_percent']] = working.apply(compute_prices, axis=1)

# 6. Category/tags remove if <50%
def keep(col):
    filled = (working[col].str.strip() != '').sum()
    return filled / len(working) >= CATEGORY_COVERAGE_THRESHOLD

if not keep('category'):
    working.drop(columns=['category'], inplace=True)

if not keep('tags'):
    working.drop(columns=['tags'], inplace=True)

# 7. Clean functional description
working['long_description'] = working['functional'].apply(clean_functional_text)

# 8. Summary
working['summary'] = working['long_description'].apply(lambda t: extractive_summary(t, SUMMARY_SENTENCES))

# 9. Lemmatized text
working['indexed_text_lemma'] = working.apply(
    lambda r: lemmatize_text(r['title'] + ' ' + r['summary'] + ' ' + r['long_description']), axis=1
)

# 10. Clean URL
working['product_url'] = working['product_url'].apply(clean_url)

# 11. Image URL rule (keep or drop)
img_frac = working['image_url'].apply(looks_like_image_url).mean()
if img_frac > 0.99:
    working.drop(columns=['image_url'], inplace=True)

# 12. Final search content
working['search_content'] = (
    working['title'] + " " + working['summary'] + " " + working['long_description']
)

# Final Export Columns (NO vendor, NO notes)
export_cols = [
    'sku','title','price_current','price_original','discount_percent',
    'stock_status','summary','long_description','search_content',
    'indexed_text_lemma','product_url'
]

export_cols = [c for c in export_cols if c in working.columns]

cleaned = working[export_cols].copy()

# Save File
cleaned.to_csv(OUTPUT_PATH, index=False)
print("Saved cleaned CSV:", OUTPUT_PATH)
print("Rows:", len(cleaned))
cleaned.head()

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Loaded: 90
Saved cleaned CSV: /kaggle/working/gangslifestyle.csv
Rows: 90


,sku,title,price_current,price_original,discount_percent,stock_status,summary,long_description,search_content,indexed_text_lemma,product_url
0,MK173,Foldable Storage Organizer Basket with Lid – T...,299.0,299.0,0.0,In Stock,Foldable Design: Maximize your space when not ...,About this Item Lightweight and Durable Materi...,Foldable Storage Organizer Basket with Lid – T...,foldable storage organizer basket with lid tra...,https://99wholesale.com/products/foldable-stor...
1,LT035,DIY Wooden Piggy Bank – Carved Coin Box for Ki...,349.0,349.0,0.0,In Stock,Popular Searches Erlik Wooden Piggy Bank for K...,Material - Wood Product Weight- 200gm Product ...,DIY Wooden Piggy Bank – Carved Coin Box for Ki...,diy wooden piggy bank carved coin box for kid ...,https://99wholesale.com/products/wooden-piggy-...
2,MK170,24 in 1 Mini Professional Hardware Tools with ...,129.0,129.0,0.0,Out of Stock,This kit equips you with the necessary tools f...,About this Item Lightweight and Durable Materi...,24 in 1 Mini Professional Hardware Tools with ...,24 in 1 mini professional hardware tool with m...,https://99wholesale.com/products/24-in-1-mini-...
3,MK081,Automatic Toothpaste Dispenser & Holder – Wall...,129.0,129.0,0.0,In Stock,Product Specifications Material: Durable ABS p...,Product Specifications Material: Durable ABS p...,Automatic Toothpaste Dispenser & Holder – Wall...,automatic toothpaste dispenser holder toothbru...,https://99wholesale.com/products/automatic-too...
4,WJI173,LED Ice Cube Light (Pack of 2 Pcs),49.0,99.0,50.5,Out of Stock,Perfect for adding a touch of flair to your pa...,Material: Plastic Product Weight: 50 Gms Produ...,LED Ice Cube Light (Pack of 2 Pcs) Perfect for...,led ice cube light pack of 2 pc perfect for ad...,https://99wholesale.com/products/led-ice-cube-...


### 99 Whole Sale CSV file

In [4]:
# FINAL PREPROCESSING PIPELINE – CLEAN VERSION
# No vendor column, no notes column, no last_scraped_date, no source_file.

import re, os, math
from datetime import datetime
import numpy as np
import pandas as pd

# NLP imports
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

# download NLP resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ---------- USER CONFIG ----------
INPUT_PATH = "/kaggle/input/arniova-studio/99 Whole Sale 14NOV2025.csv"
OUTPUT_PATH = "/kaggle/working/99wholesale_collections_trending-product.csv"
CATEGORY_COVERAGE_THRESHOLD = 0.5
VARIANT_COLOR_COMBINE = True
SUMMARY_SENTENCES = 1
# ------------------------------------

# Helpers
_currency_re = re.compile(r'[^\d.,\-]+')
def parse_price(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() in ['nan','none','null']:
        return np.nan
    s = _currency_re.sub('', s).replace(',', '')
    try: return float(s)
    except:
        nums = re.findall(r'[-+]?\d*\.\d+|\d+', s)
        return float(nums[0]) if nums else np.nan

def normalize_stock(v):
    if pd.isna(v): return 'Unknown'
    s = str(v).strip().lower()
    if s in ['true','yes','1','in stock','available','instock']: return 'In Stock'
    if s in ['false','no','0','out of stock','sold out','not available']: return 'Out of Stock'
    return 'Unknown'

MARKETING_WORDS = {'buy now','best','new','free shipping','hot','sale','discount','offer','trending'}
def clean_title(t):
    if pd.isna(t): return ''
    s = re.sub(r'\s+', ' ', str(t).strip())
    for w in MARKETING_WORDS:
        s = re.sub(r'(?i)\\b' + re.escape(w) + r'\\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()

COLOR_WORDS = {'black','white','red','blue','green','yellow','pink','orange','purple','brown','grey','gray','silver','gold','navy'}
def variant_looks_like_color(v):
    if pd.isna(v) or str(v).strip() == '': return False
    parts = re.split(r'[,/;|-]+', str(v).lower())
    return any(p.strip() in COLOR_WORDS for p in parts)

def clean_functional_text(txt):
    if pd.isna(txt): return ''
    s = re.sub(r'<[^>]+>', ' ', str(txt))
    s = re.sub(r'[\r\n\t]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    words = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(w.lower()) for w in words if w.isalnum())

def extractive_summary(text, n_sentences=1):
    if not text.strip(): return ''
    sents = sent_tokenize(text)
    if len(sents) <= n_sentences: return ' '.join(sents)
    try:
        vec = TfidfVectorizer(stop_words='english')
        X = vec.fit_transform(sents)
        centroid = X.sum(axis=0)
        scores = X.dot(centroid.T).A.ravel()
        idx = scores.argsort()[::-1][:n_sentences]
        idx = sorted(idx)
        return ' '.join(sents[i].strip() for i in idx)
    except:
        return sents[0]

def clean_url(u):
    if pd.isna(u): return ''
    s = str(u).strip()
    return re.sub(r'[\?&]$', '', s)

def looks_like_image_url(u):
    if pd.isna(u): return False
    return bool(re.search(r'\.(jpg|jpeg|png|webp|gif)$', str(u), flags=re.I))

# ---------- Load CSV ----------
df = pd.read_csv(INPUT_PATH, dtype=str)
print("Loaded:", len(df))

# Exact column mapping
col = {
    "title": "Product Name",
    "variant": "Variant Name",
    "sku": "SKU",
    "instock": "In Stock?",
    "price": "Price",
    "original_price": "Original Price",
    "discount_info": "Discount Info",
    "category": "Category",
    "tags": "Tags",
    "functional": "Functional Details",
    "product_url": "Link",
    "image_url": "Main Image URL"
}

working = pd.DataFrame()
for k, v in col.items():
    working[k] = df[v].fillna('') if v in df.columns else ''

# 1. Clean title
working['title'] = working['title'].apply(clean_title)

# 2. Combine variant for colors
if VARIANT_COLOR_COMBINE:
    working['title'] = working.apply(
        lambda r: f"{r['title']} (Color: {r['variant']})" 
        if variant_looks_like_color(r['variant']) else r['title'], axis=1
    )

# 3. SKU fallback
working['sku'] = working['sku'].astype(str).str.strip()
missing = working['sku'] == ''
working.loc[missing, 'sku'] = [f"MISSINGSKU_{i}" for i in range(1, missing.sum()+1)]

# 4. Stock normalize
working['stock_status'] = working['instock'].apply(normalize_stock)

# 5. PRICE + ORIGINAL PRICE FIX (YOU REQUESTED)
working['price_parsed'] = working['price'].apply(parse_price)
working['original_price_parsed'] = working['original_price'].apply(parse_price)

def compute_prices(r):
    p = r['price_parsed']
    o = r['original_price_parsed']

    # If original missing or zero → set original = price
    if pd.isna(o) or o == 0:
        o = p

    # If price missing → set price = original
    if pd.isna(p) and not pd.isna(o):
        p = o

    # Compute discount
    if pd.isna(p) or pd.isna(o):
        disc = np.nan
    else:
        disc = 0 if o == p else round((o - p) / o * 100, 1) if o > p else 0

    return pd.Series([p, o, disc])

working[['price_current', 'price_original', 'discount_percent']] = working.apply(compute_prices, axis=1)

# 6. Category/tags remove if <50%
def keep(col):
    filled = (working[col].str.strip() != '').sum()
    return filled / len(working) >= CATEGORY_COVERAGE_THRESHOLD

if not keep('category'):
    working.drop(columns=['category'], inplace=True)

if not keep('tags'):
    working.drop(columns=['tags'], inplace=True)

# 7. Clean functional description
working['long_description'] = working['functional'].apply(clean_functional_text)

# 8. Summary
working['summary'] = working['long_description'].apply(lambda t: extractive_summary(t, SUMMARY_SENTENCES))

# 9. Lemmatized text
working['indexed_text_lemma'] = working.apply(
    lambda r: lemmatize_text(r['title'] + ' ' + r['summary'] + ' ' + r['long_description']), axis=1
)

# 10. Clean URL
working['product_url'] = working['product_url'].apply(clean_url)

# 11. Image URL rule (keep or drop)
img_frac = working['image_url'].apply(looks_like_image_url).mean()
if img_frac > 0.99:
    working.drop(columns=['image_url'], inplace=True)

# 12. Final search content
working['search_content'] = (
    working['title'] + " " + working['summary'] + " " + working['long_description']
)

# Final Export Columns (NO vendor, NO notes)
export_cols = [
    'sku','title','price_current','price_original','discount_percent',
    'stock_status','summary','long_description','search_content',
    'indexed_text_lemma','product_url'
]

export_cols = [c for c in export_cols if c in working.columns]

cleaned = working[export_cols].copy()

# Save File
cleaned.to_csv(OUTPUT_PATH, index=False)
print("Saved cleaned CSV:", OUTPUT_PATH)
print("Rows:", len(cleaned))
cleaned.head()

Loaded: 66


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Saved cleaned CSV: /kaggle/working/99wholesale_collections_trending-product.csv
Rows: 66


,sku,title,price_current,price_original,discount_percent,stock_status,summary,long_description,search_content,indexed_text_lemma,product_url
0,GANGS2028,The StackLine Luggage - Set of 3,11999.0,32999.0,63.6,In Stock,"Crafted with meticulous attention to detail, t...",Introducing the epitome of style and functiona...,The StackLine Luggage - Set of 3 Crafted with ...,the stackline luggage set of 3 crafted with me...,https://99wholesale.com/products/the-stackline...
1,GANGS2029,The StackLine Luggage - Set of 3,11999.0,32999.0,63.6,In Stock,"Crafted with meticulous attention to detail, t...",Introducing the epitome of style and functiona...,The StackLine Luggage - Set of 3 Crafted with ...,the stackline luggage set of 3 crafted with me...,https://99wholesale.com/products/the-stackline...
2,GANGS2030,The StackLine Luggage - Set of 3,11999.0,32999.0,63.6,In Stock,"Crafted with meticulous attention to detail, t...",Introducing the epitome of style and functiona...,The StackLine Luggage - Set of 3 Crafted with ...,the stackline luggage set of 3 crafted with me...,https://99wholesale.com/products/the-stackline...
3,GANGS2036,The VelocityTrunk Luggage - Set of 3,13999.0,37999.0,63.2,In Stock,"Crafted with meticulous attention to detail, t...",Introducing the epitome of style and functiona...,The VelocityTrunk Luggage - Set of 3 Crafted w...,the velocitytrunk luggage set of 3 crafted wit...,https://99wholesale.com/products/the-velocityt...
4,GANGS2034,The VelocityTrunk Luggage - Set of 3,13999.0,37999.0,63.2,In Stock,"Crafted with meticulous attention to detail, t...",Introducing the epitome of style and functiona...,The VelocityTrunk Luggage - Set of 3 Crafted w...,the velocitytrunk luggage set of 3 crafted wit...,https://99wholesale.com/products/the-velocityt...


## Combined Code - Web Scrapping + Preprocessing

### GangsLifeStyle

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import os
from datetime import datetime
from urllib.parse import urlparse, urljoin
# --- Configuration Constants ---
START_URL = "https://gangslifestyle.com/" # Main website to start scraping from
BASE_SAVE_DIR = '/content/' # Changed this to '/content/' as requested
REQUEST_DELAY = 1.0  # seconds
TIMEOUT = 15  # seconds
USER_AGENT = 'Mozilla/5.0 (compatible; DataScraper/1.0)'
MAX_PAGES = 2000 # Max pages to paginate through for a single collection (safety limit)
# --- Custom URL to Name Function ---
def url_to_name(url):
    parsed = urlparse(url)
    netloc = parsed.netloc
    # Remove common subdomains like 'www', 'm', 'app', 'shop'
    parts = netloc.split(".")
    if parts[0] in ["www", "m", "app", "shop"]:
        parts = parts[1:]
    domain = parts[0]
    # Get path parts and filter out empty strings
    path_parts = [p for p in parsed.path.split('/') if p]
    if path_parts:
        # Combine domain with path parts, replacing special characters for safety
        safe_path = "_".join(path_parts).replace('-', '_').replace('.', '_')
        return domain + "_" + safe_path
    else:
        return domain
print("Libraries imported, constants defined, and url_to_name function created.")
# --- Get Collection URLs ---
collection_urls = set()
print(f"\nFetching collection links from: {START_URL}")
try:
    response = requests.get(START_URL, headers={'User-Agent': USER_AGENT}, timeout=TIMEOUT)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'html.parser')
    start_url_parsed = urlparse(START_URL)
    start_domain = start_url_parsed.netloc
    for a_tag in soup.find_all('a', href=True):
        link = urljoin(START_URL, a_tag['href'])
        parsed_link = urlparse(link)
        # Filter for internal links with the same domain and containing '/collections/'
        if parsed_link.netloc == start_domain and '/collections/' in parsed_link.path:
            # Normalize the URL to remove query parameters and fragments for uniqueness
            clean_link = parsed_link.scheme + "://" + parsed_link.netloc + parsed_link.path.split('?')[0].split('#')[0]
            if clean_link.endswith('/'):
                clean_link = clean_link[:-1]
            collection_urls.add(clean_link)
    print(f"Found {len(collection_urls)} potential unique collection URLs.")
    print("First 5 collection URLs:")
    for i, url in enumerate(list(collection_urls)[:5]):
        print(f"  {i+1}. {url}")
except requests.exceptions.RequestException as e:
    print(f"Error making request to {START_URL}: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    time.sleep(REQUEST_DELAY)
# --- Scrape Each Collection and Save Data ---
all_collections_data = []
# Ensure the base save directory exists
os.makedirs(BASE_SAVE_DIR, exist_ok=True)
print(f"\nEnsured base save directory exists: {BASE_SAVE_DIR}")
print(f"Starting scrape for {len(collection_urls)} collections...")
# Loop through each unique collection URL
for i, collection_url in enumerate(sorted(list(collection_urls))):
    collection_json_endpoint = f"{collection_url}/products.json"
    collection_name_for_logging = url_to_name(collection_url)
    print(f"\nProcessing collection {i + 1}/{len(collection_urls)}: {collection_name_for_logging}")
    print(f"  JSON Endpoint: {collection_json_endpoint}")
    current_collection_variants_data = []
    page = 1
    # Iterate through paginated data for the current collection
    while True:
        json_url = f'{collection_json_endpoint}?page={page}&limit=250'
        print(f"  Fetching page {page} for {collection_name_for_logging}: {json_url}")
        try:
            response = requests.get(json_url, headers={'User-Agent': USER_AGENT}, timeout=TIMEOUT)
            response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
            data = response.json()
            # Break if no more products are found on this page
            if 'products' not in data or not data['products']:
                print(f"  No more products found for {collection_name_for_logging} on page {page}. Stopping pagination.")
                break
            products = data['products']
            print(f"  Found {len(products)} products on page {page}.")
            for product in products:
                try:
                    # A: Get Product-Level Data
                    product_title = product.get('title', 'N/A')
                    handle = product.get('handle')
                    vendor = product.get('vendor', 'N/A')
                    category = product.get('product_type', 'N/A')
                    # Clean HTML description
                    raw_html = product.get('body_html', '')
                    if raw_html:
                        soup = BeautifulSoup(raw_html, 'html.parser')
                        functional_details = soup.get_text(separator=' ', strip=True)
                    else:
                        functional_details = "N/A"
                    # Get tags
                    tags = ', '.join(product.get('tags', []))
                    # Get main image
                    main_image_url = "N/A"
                    if product.get('images'):
                        main_image_url = product['images'][0]['src']
                    # B: Loop through EACH variant for this product
                    for variant in product.get('variants', []):
                        variant_title = variant.get('title', 'N/A')
                        variant_id = variant.get('id')
                        price = variant.get('price', 'N/A')
                        original_price = variant.get('compare_at_price') # This is the "was" price
                        sku = variant.get('sku', 'N/A')
                        available = variant.get('available', False) # True/False
                        # Precise Link for the variant
                        parsed_collection_url = urlparse(collection_url)
                        base_shop_url = f"{parsed_collection_url.scheme}://{parsed_collection_url.netloc}"
                        link = f"{base_shop_url}/products/{handle}?variant={variant_id}"
                        # Discount logic
                        discount_info = "No Discount"
                        try:
                            p = float(price)
                            op = float(original_price) if original_price else None
                            if op is not None and op > p:
                                discount_info = f"Was {original_price}"
                        except ValueError:
                            pass # Keep "No Discount" if prices are not numeric
                        # Append one row for THIS variant
                        current_collection_variants_data.append({
                            'Collection URL': collection_url,
                            'Product Name': product_title,
                            'Variant Name': variant_title,
                            'SKU': sku,
                            'In Stock?': available,
                            'Price': price,
                            'Original Price': original_price if original_price else "N/A",
                            'Discount Info': discount_info,
                            'Vendor (Brand)': vendor,
                            'Category': category,
                            'Tags': tags,
                            'Functional Details': functional_details,
                            'Link': link,
                            'Main Image URL': main_image_url
                        })
                except Exception as e:
                    print(f"  Error parsing product '{product.get('title', 'Unknown')}' in {collection_name_for_logging}: {e}")
            page += 1
            time.sleep(REQUEST_DELAY) # Polite delay
        except requests.exceptions.HTTPError as e:
            print(f"  HTTP Error for {collection_name_for_logging} on page {page}: {e}. Status code: {e.response.status_code}. Stopping pagination for this collection.")
            break
        except requests.exceptions.RequestException as e:
            print(f"  Request Error for {collection_name_for_logging} on page {page}: {e}. Stopping pagination for this collection.")
            break
        except Exception as e:
            print(f"  An unexpected error occurred for {collection_name_for_logging} on page {page}: {e}. Stopping pagination for this collection.")
            break
    # After pagination for a single collection, append its data to the overall list
    if current_collection_variants_data:
        print(f"  Collected {len(current_collection_variants_data)} variants for {collection_name_for_logging}.")
        all_collections_data.extend(current_collection_variants_data)
    else:
        print(f"  No variants collected for {collection_name_for_logging}.")
print(f"\nFinished scraping all collections. Total variants collected: {len(all_collections_data)}")
print("\nSaving scraped data to individual collection CSV files...")
# Create a Pandas DataFrame from all collected data
if all_collections_data:
    df_all_variants = pd.DataFrame(all_collections_data)
    # Get the base name for the root directory (e.g., '99wholesale')
    base_url_name = url_to_name(START_URL) # Using START_URL for the top-level directory name
    site_specific_save_dir = os.path.join(BASE_SAVE_DIR, base_url_name)
    # Create the site-specific directory if it doesn't exist
    os.makedirs(site_specific_save_dir, exist_ok=True)
    print(f"Ensured site-specific save directory exists: {site_specific_save_dir}")
    # Group by 'Collection URL' and save each group to a separate CSV
    for collection_url, group_df in df_all_variants.groupby('Collection URL'):
        # Use the url_to_name function for the collection URL to create a safe filename
        collection_filename_base = url_to_name(collection_url)
        # Construct the full path for the CSV file
        csv_filename = os.path.join(site_specific_save_dir, f'{collection_filename_base}.csv')
        group_df.to_csv(csv_filename, index=False, encoding='utf-8')
        print(f"  Saved {len(group_df)} variants for '{collection_filename_base}' to '{csv_filename}'.")
else:
    print("No data was collected to save.")
print("Data saving complete.")
print("\nAll collections have been scraped and saved to individual CSV files within the '99wholesale' directory.")
print(f"You can find the data in the directory: {site_specific_save_dir}")

### 99WholeSale

In [1]:
"""
Combined: Shopify JSON scraper (variants) + preprocessing pipeline (clean + export)

How to use:
- Run this script with Python 3.8+ (pandas, requests, beautifulsoup4, nltk, scikit-learn installed).
- It will fetch products from the store endpoint, build a variants dataframe,
  run the cleaning pipeline in memory, and write:
    - raw output: raw_variants_output.csv
    - cleaned output: cleaned_products.csv

Adjust USER CONFIG and BASE_URL as needed.
"""

import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import re
import os
import math
from datetime import datetime
import numpy as np

# NLP imports
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

# URL to Name
from urllib.parse import urlparse

def url_to_name(url):
    parsed = urlparse(url)

    # ---- Extract domain safely ----
    netloc = parsed.netloc

    # Remove common subdomains like www, m, shop, app, etc.
    parts = netloc.split(".")
    if parts[0] in ["www", "m", "app", "shop"]:
        parts = parts[1:]  # remove prefix
    
    # domain name (before TLD)
    domain = parts[0]

    # ---- Extract path parts ----
    path_parts = [p for p in parsed.path.split('/') if p]

    # ---- Build final name ----
    if path_parts:
        return domain + "_" + "_".join(path_parts)
    else:
        return domain

# ----------------- USER CONFIG -----------------
BASE_URL = "https://99wholesale.com/collections/trending-product"
JSON_ENDPOINT = f"{BASE_URL}/products.json"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)'
}
name = url_to_name(BASE_URL);
OUTPUT_CSV = f"{name}.csv"    

# Preprocessing config (from your Code 2)
CATEGORY_COVERAGE_THRESHOLD = 0.5
VARIANT_COLOR_COMBINE = True
SUMMARY_SENTENCES = 1
# ------------------------------------------------

# Ensure resources for NLTK (will check & download if needed)
nltk_data = ['punkt', 'stopwords', 'wordnet', 'omw-1.4']
for r in nltk_data:
    try:
        nltk.data.find(r)
    except Exception:
        nltk.download(r)

# ----------------- SCRAPER (variant-mode) -----------------
def scrape_shopify_products(json_endpoint=JSON_ENDPOINT, headers=HEADERS, sleep_sec=1):
    all_variants_data = []
    page = 1
    print("--- Starting Scrape (Shopify JSON - Detailed Variant Mode) ---")
    while True:
        url = f"{json_endpoint}?page={page}&limit=250"
        print(f"Fetching Page {page}: {url}")
        try:
            response = requests.get(url, headers=headers, timeout=20)
            if response.status_code != 200:
                print(f"  Status code {response.status_code}. Stopping.")
                break
            data = response.json()
            if 'products' not in data or not data['products']:
                print("  No more products found in JSON. Crawl finished.")
                break

            products = data['products']
            print(f"  Found {len(products)} products on page {page} ...")

            for product in products:
                try:
                    product_title = product.get('title', 'N/A')
                    handle = product.get('handle', '')
                    vendor = product.get('vendor', 'N/A')
                    category = product.get('product_type', 'N/A')

                    raw_html = product.get('body_html', '')
                    if raw_html:
                        soup = BeautifulSoup(raw_html, 'html.parser')
                        functional_details = soup.get_text(separator=' ', strip=True)
                    else:
                        functional_details = "N/A"

                    # tags: product['tags'] may be a string or list depending on store — handle both
                    tags_val = product.get('tags', '')
                    if isinstance(tags_val, list):
                        tags = ', '.join(tags_val)
                    else:
                        # sometimes Shopify returns a single string with commas
                        tags = tags_val if tags_val else ''

                    main_image_url = "N/A"
                    if product.get('images'):
                        # images is list of dicts with 'src'
                        try:
                            if isinstance(product['images'], list) and product['images']:
                                main_image_url = product['images'][0].get('src', 'N/A')
                            else:
                                main_image_url = product['images']
                        except Exception:
                            main_image_url = "N/A"

                    for variant in product.get('variants', []):
                        variant_title = variant.get('title', 'N/A')
                        variant_id = variant.get('id')
                        price = variant.get('price', 'N/A')
                        original_price = variant.get('compare_at_price')  # "was" price
                        sku = variant.get('sku', 'N/A')
                        available = variant.get('available', False)

                        link = f"{BASE_URL}/products/{handle}?variant={variant_id}" if handle and variant_id else f"{BASE_URL}"

                        if original_price and original_price != price:
                            discount_info = f"Was {original_price}"
                        else:
                            discount_info = "No Discount"

                        all_variants_data.append({
                            'Product Name': product_title,
                            'Variant Name': variant_title,
                            'SKU': sku,
                            'In Stock?': available,
                            'Price': price,
                            'Original Price': original_price if original_price else "",
                            'Discount Info': discount_info,
                            'Vendor (Brand)': vendor,
                            'Category': category,
                            'Tags': tags,
                            'Functional Details': functional_details,
                            'Link': link,
                            'Main Image URL': main_image_url
                        })

                except Exception as e:
                    print(f"  Error parsing product '{product.get('title', 'unknown')}': {e}")

            page += 1
            time.sleep(sleep_sec)

        except Exception as e:
            print(f"  Error fetching URL or parsing JSON: {e}")
            break

    return pd.DataFrame(all_variants_data)

# ----------------- PREPROCESSING PIPELINE (adapted from Code 2) -----------------
# Helpers (kept from your code)
_currency_re = re.compile(r'[^\d.,\-]+')
def parse_price(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() in ['nan','none','null']:
        return np.nan
    s = _currency_re.sub('', s).replace(',', '')
    try:
        return float(s)
    except:
        nums = re.findall(r'[-+]?\d*\.\d+|\d+', s)
        return float(nums[0]) if nums else np.nan

def normalize_stock(v):
    if pd.isna(v): return 'Unknown'
    s = str(v).strip().lower()
    if s in ['true','yes','1','in stock','available','instock', 'True', 'TRUE', '1']: return 'In Stock'
    if s in ['false','no','0','out of stock','sold out','not available', 'False', 'FALSE', '0']: return 'Out of Stock'
    return 'Unknown'

MARKETING_WORDS = {'buy now','best','new','free shipping','hot','sale','discount','offer','trending'}
def clean_title(t):
    if pd.isna(t): return ''
    s = re.sub(r'\s+', ' ', str(t).strip())
    for w in MARKETING_WORDS:
        # case-insensitive whole-word removal
        s = re.sub(r'(?i)\b' + re.escape(w) + r'\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()

COLOR_WORDS = {'black','white','red','blue','green','yellow','pink','orange','purple','brown','grey','gray','silver','gold','navy'}
def variant_looks_like_color(v):
    if pd.isna(v) or str(v).strip() == '': return False
    parts = re.split(r'[,/;|-]+', str(v).lower())
    return any(p.strip() in COLOR_WORDS for p in parts)

def clean_functional_text(txt):
    if pd.isna(txt): return ''
    s = re.sub(r'<[^>]+>', ' ', str(txt))
    s = re.sub(r'[\r\n\t]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    # safe guard: if text empty
    if not isinstance(text, str) or text.strip() == '':
        return ''
    words = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(w.lower()) for w in words if w.isalnum())

def extractive_summary(text, n_sentences=1):
    if not isinstance(text, str) or not text.strip():
        return ''
    sents = sent_tokenize(text)
    if len(sents) <= n_sentences:
        return ' '.join(sents)
    try:
        vec = TfidfVectorizer(stop_words='english')
        X = vec.fit_transform(sents)
        centroid = X.sum(axis=0)
        scores = X.dot(centroid.T).A.ravel()
        idx = scores.argsort()[::-1][:n_sentences]
        idx = sorted(idx)
        return ' '.join(sents[i].strip() for i in idx)
    except Exception:
        return sents[0]

def clean_url(u):
    if pd.isna(u): return ''
    s = str(u).strip()
    return re.sub(r'[\?&]$', '', s)

def looks_like_image_url(u):
    if pd.isna(u): return False
    return bool(re.search(r'\.(jpg|jpeg|png|webp|gif)$', str(u), flags=re.I))


def preprocess_variants_df(df_raw):
    """
    Accepts raw DataFrame similar to Code 1 output and returns cleaned DataFrame similar to Code 2 output.
    """
    if df_raw is None or df_raw.shape[0] == 0:
        print("No raw data to preprocess.")
        return pd.DataFrame()

    df = df_raw.copy()

    # Standardize column names if needed (allow both exact names and a few variants)
    # Map expected input columns to short keys used inside pipeline.
    col_map = {
        'Product Name': 'title',
        'Variant Name': 'variant',
        'SKU': 'sku',
        'In Stock?': 'instock',
        'Price': 'price',
        'Original Price': 'original_price',
        'Discount Info': 'discount_info',
        'Category': 'category',
        'Tags': 'tags',
        'Functional Details': 'functional',
        'Link': 'product_url',
        'Main Image URL': 'image_url'
    }

    # Create working frame with keys from col_map
    working = pd.DataFrame()
    for short, long in col_map.items():
        if short in df.columns:
            working[long] = df[short].fillna('')
        else:
            # if expected column not present, create empty series
            working[long] = ''

    # 1. Clean title
    working['title'] = working['title'].apply(clean_title)

    # 2. Combine variant into title when variant looks like a color
    if VARIANT_COLOR_COMBINE:
        # ensure we have 'variant' present
        working['variant'] = working.get('variant', '')
        working['title'] = working.apply(
            lambda r: f"{r['title']} (Color: {r['variant']})" if variant_looks_like_color(r['variant']) else r['title'],
            axis=1
        )

    # 3. SKU fallback
    working['sku'] = working['sku'].astype(str).str.strip()
    missing = working['sku'] == ''
    if missing.any():
        working.loc[missing, 'sku'] = [f"MISSINGSKU_{i}" for i in range(1, missing.sum() + 1)]

    # 4. Stock normalize
    working['stock_status'] = working['instock'].apply(normalize_stock)

    # 5. Price parsing
    working['price_parsed'] = working['price'].apply(parse_price)
    working['original_price_parsed'] = working['original_price'].apply(parse_price)

    def compute_prices(r):
        p = r['price_parsed']
        o = r['original_price_parsed']
        if pd.isna(o) or o == 0:
            o = p
        if pd.isna(p) and not pd.isna(o):
            p = o
        if pd.isna(p) or pd.isna(o):
            disc = np.nan
        else:
            disc = 0 if o == p else round((o - p) / o * 100, 1) if o > p else 0
        return pd.Series([p, o, disc])

    working[['price_current', 'price_original', 'discount_percent']] = working.apply(compute_prices, axis=1)

    # 6. Category/tags remove if < threshold
    def keep(col):
        filled = (working[col].astype(str).str.strip() != '').sum()
        return (filled / len(working)) >= CATEGORY_COVERAGE_THRESHOLD

    if not keep('category'):
        if 'category' in working.columns:
            working.drop(columns=['category'], inplace=True)

    if not keep('tags'):
        if 'tags' in working.columns:
            working.drop(columns=['tags'], inplace=True)

    # 7. Clean functional description
    working['long_description'] = working['functional'].apply(clean_functional_text)

    # 8. Summary
    working['summary'] = working['long_description'].apply(lambda t: extractive_summary(t, SUMMARY_SENTENCES))

    # 9. Lemmatized text (safe-guard for heavy computation)
    working['indexed_text_lemma'] = working.apply(
        lambda r: lemmatize_text(str(r.get('title','')) + ' ' + str(r.get('summary','')) + ' ' + str(r.get('long_description',''))),
        axis=1
    )

    # 10. Clean URL
    working['product_url'] = working['product_url'].apply(clean_url)

    # 11. Image URL rule (keep or drop)
    img_frac = working['image_url'].apply(looks_like_image_url).mean()
    if 'image_url' in working.columns and img_frac > 0.99:
        working.drop(columns=['image_url'], inplace=True)

    # 12. Final search content
    working['search_content'] = (
        working.get('title', '') + " " + working.get('summary', '') + " " + working.get('long_description', '')
    )

    # Final export columns (NO vendor, NO notes)
    export_cols = [
        'sku','title','price_current','price_original','discount_percent',
        'stock_status','summary','long_description','search_content',
        'indexed_text_lemma','product_url'
    ]
    export_cols = [c for c in export_cols if c in working.columns]
    cleaned = working[export_cols].copy()

    return cleaned

# ----------------- Main flow -----------------
def main():
    # 1) Scrape
    raw_df = scrape_shopify_products()

    if raw_df is None or raw_df.shape[0] == 0:
        print("No data scraped. Exiting.")
        return

    # Save raw CSV (optional)
    try:
        raw_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
        print("Saved raw variants CSV:", OUTPUT_CSV)
    except Exception as e:
        print("Could not save raw CSV:", e)

    # 2) Preprocess
    cleaned = preprocess_variants_df(raw_df)

    # 3) Save cleaned CSV
    try:
        cleaned.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
        print("Saved cleaned CSV:", OUTPUT_CSV)
        print("Rows in cleaned:", len(cleaned))
    except Exception as e:
        print("Could not save cleaned CSV:", e)

    # show head for quick check
    print("\n--- Raw head ---")
    print(raw_df.head())
    print("\n--- Cleaned head ---")
    print(cleaned.head())

if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pavan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pavan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\pavan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\pavan\AppData\Roaming\nltk_data...


--- Starting Scrape (Shopify JSON - Detailed Variant Mode) ---
Fetching Page 1: https://99wholesale.com/collections/trending-product/products.json?page=1&limit=250
  Status code 403. Stopping.
No data scraped. Exiting.


[nltk_data]   Package omw-1.4 is already up-to-date!


## Query Response Generation - Version 1

### GangsLifeStyle

In [7]:
# === QUERY HANDLER FOR KAGGLE NOTEBOOK ===
# Load a CSV, build TF-IDF, ask user for question, get answer + links.

import os
import re
import json
import time
from io import BytesIO

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional NLTK pieces used for sentence splitting (same as your API code)
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer

# minimal nltk assets required (download if missing)
_nltk_needed = ['punkt','wordnet','omw-1.4']
for r in _nltk_needed:
    try:
        nltk.data.find(r)
    except Exception:
        try:
            nltk.download(r, quiet=True)
        except Exception:
            pass

# ---------- CONFIG ----------
CSV_PATH = "/kaggle/working/gangslifestyle.csv"   # <-- change to your cleaned CSV
SIMILARITY_THRESHOLD = 0.30
TOP_K_DEFAULT = 3
OUTPUT_DIR = "/kaggle/working"                           # where JSON responses will be saved

# ---------- HELPERS ----------
IRRELEVANT_KEYWORDS = [
    "gym","dumbbell","exercise","workout","recipe","food","cooking",
    "weather","news","politics","relationship","doctor","medicine",
    "math","code","python","java","cpp"
]
_irrelevant_regex = re.compile(r'\b(' + r'|'.join(re.escape(w) for w in IRRELEVANT_KEYWORDS) + r')\b', flags=re.I)

def is_irrelevant(query: str) -> bool:
    if not query or str(query).strip() == "":
        return True
    return bool(_irrelevant_regex.search(query))

def safe_jsonify(obj):
    """Convert numpy / pandas types to JSON-serializable types."""
    if isinstance(obj, dict):
        return {safe_jsonify(k): safe_jsonify(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [safe_jsonify(x) for x in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return obj

def choose_text_field(df: pd.DataFrame) -> str:
    """Pick the best field to index for retrieval (prefer indexed_text_lemma, then search_content, else first text-like)."""
    if 'indexed_text_lemma' in df.columns and df['indexed_text_lemma'].astype(str).str.strip().any():
        return 'indexed_text_lemma'
    if 'search_content' in df.columns and df['search_content'].astype(str).str.strip().any():
        return 'search_content'
    for candidate in ['long_description','description','title']:
        if candidate in df.columns:
            return candidate
    return df.columns[0]

def build_tfidf_index(df: pd.DataFrame, field: str):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
    mat = vec.fit_transform(df[field].astype(str).fillna('').values)
    return vec, mat

def query_search_from_index(query: str,
                             df: pd.DataFrame,
                             vectorizer: TfidfVectorizer,
                             tfidf_matrix,
                             text_field: str,
                             top_k: int):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).ravel()
    idxs = sims.argsort()[::-1][:top_k]
    rows = []
    for i in idxs:
        rows.append({
            'clean_text': df.iloc[i].get('search_content', '') or df.iloc[i].get(text_field, ''),
            'product_title': df.iloc[i].get('title',''),
            'url': df.iloc[i].get('product_url','') or df.iloc[i].get('url',''),
            'score': float(sims[i]),
            'sku': df.iloc[i].get('sku','')
        })
    return pd.DataFrame(rows)

def extractive_summary_from_retrieved(retrieved_df, top_k=TOP_K_DEFAULT):
    if retrieved_df is None or retrieved_df.empty:
        return ""
    merged = " ".join(str(x) for x in retrieved_df['clean_text'].astype(str).head(top_k).tolist())
    merged = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', ' ', merged)
    sents = re.split(r'(?<=[.!?])\s+', merged)
    return " ".join(sents[:2]).strip()

def handle_query_with_df(df: pd.DataFrame, query: str, top_k: int = TOP_K_DEFAULT):
    """
    df: dataframe loaded from CSV
    query: user query string
    returns: dict {query, top_results, final_answer, product_links}
    """
    fallback_msg = "Sorry, I couldn't answer that. I can assist you with product, website, business or item-related queries."

    if is_irrelevant(query):
        return {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}

    text_field = choose_text_field(df)
    vectorizer, tfidf_matrix = build_tfidf_index(df, text_field)

    retrieved = query_search_from_index(query=query,
                                        df=df,
                                        vectorizer=vectorizer,
                                        tfidf_matrix=tfidf_matrix,
                                        text_field=text_field,
                                        top_k=top_k)
    if retrieved.empty:
        return {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}

    best_score = float(retrieved['score'].max())
    if best_score < SIMILARITY_THRESHOLD:
        return {
            "query": query,
            "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
            "final_answer": fallback_msg,
            "product_links": []
        }

    final_answer = extractive_summary_from_retrieved(retrieved, top_k=top_k) or fallback_msg
    product_links = [r.get('url','') for _, r in retrieved.head(top_k).iterrows() if r.get('url')]

    out = {
        "query": query,
        "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
        "final_answer": final_answer,
        "product_links": product_links
    }
    return out

def save_result_as_json(result: dict, base_name: str = "query_response"):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = int(time.time())
    path = os.path.join(OUTPUT_DIR, f"{base_name}_{ts}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_jsonify(result), f, indent=2, ensure_ascii=False)
    return path

# ---------- LOAD CSV ONCE ----------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, dtype=str).fillna('')
print(f"Loaded CSV with {len(df)} rows from {CSV_PATH}")

# ---------- INTERACTIVE QUERY ----------
# You can change this to a fixed string instead of input() if you prefer.
user_query = input("Enter your question about the products: ").strip()

result = handle_query_with_df(df, user_query, top_k=TOP_K_DEFAULT)

print("\n--- Final Answer ---")
print(result["final_answer"])
print("\nProduct links:")
for link in result["product_links"]:
    print(" -", link)

# Save to JSON file (optional)
saved_path = save_result_as_json(result, base_name="query_response")
print(f"\nResponse saved to: {saved_path}")

# If you want to inspect the whole JSON here:
import pprint; pprint.pp(result)

Loaded CSV with 66 rows from /kaggle/working/gangslifestyle.csv


Enter your question about the products:  luggage bags



--- Final Answer ---
The StackLine Luggage - Set of 3 Crafted with meticulous attention to detail, this luggage is designed to be your ultimate companion. Introducing the epitome of style and functionality   the Premium Gang s Backpack   Luggage.

Product links:
 - https://gangslifestyle.com//products/the-stackline-luggage-set-of-3?variant=51133400875320
 - https://gangslifestyle.com//products/the-stackline-luggage-set-of-3?variant=51133400940856
 - https://gangslifestyle.com//products/the-stackline-luggage-set-of-3?variant=51133400908088

Response saved to: /kaggle/working/query_response_1763998010.json
{'query': 'luggage bags',
 'top_results': [{'clean_text': 'The StackLine Luggage - Set of 3 Crafted with '
                                'meticulous attention to detail, this luggage '
                                'is designed to be your ultimate companion. '
                                'Introducing the epitome of style and '
                                "functionality – t

### 99WholeSale

In [8]:
# === QUERY HANDLER FOR KAGGLE NOTEBOOK ===
# Load a CSV, build TF-IDF, ask user for question, get answer + links.

import os
import re
import json
import time
from io import BytesIO

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional NLTK pieces used for sentence splitting (same as your API code)
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer

# minimal nltk assets required (download if missing)
_nltk_needed = ['punkt','wordnet','omw-1.4']
for r in _nltk_needed:
    try:
        nltk.data.find(r)
    except Exception:
        try:
            nltk.download(r, quiet=True)
        except Exception:
            pass

# ---------- CONFIG ----------
CSV_PATH = "/kaggle/working/99wholesale_collections_trending-product.csv"   # <-- change to your cleaned CSV
SIMILARITY_THRESHOLD = 0.30
TOP_K_DEFAULT = 3
OUTPUT_DIR = "/kaggle/working"                           # where JSON responses will be saved

# ---------- HELPERS ----------
IRRELEVANT_KEYWORDS = [
    "gym","dumbbell","exercise","workout","recipe","food","cooking",
    "weather","news","politics","relationship","doctor","medicine",
    "math","code","python","java","cpp"
]
_irrelevant_regex = re.compile(r'\b(' + r'|'.join(re.escape(w) for w in IRRELEVANT_KEYWORDS) + r')\b', flags=re.I)

def is_irrelevant(query: str) -> bool:
    if not query or str(query).strip() == "":
        return True
    return bool(_irrelevant_regex.search(query))

def safe_jsonify(obj):
    """Convert numpy / pandas types to JSON-serializable types."""
    if isinstance(obj, dict):
        return {safe_jsonify(k): safe_jsonify(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [safe_jsonify(x) for x in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return obj

def choose_text_field(df: pd.DataFrame) -> str:
    """Pick the best field to index for retrieval (prefer indexed_text_lemma, then search_content, else first text-like)."""
    if 'indexed_text_lemma' in df.columns and df['indexed_text_lemma'].astype(str).str.strip().any():
        return 'indexed_text_lemma'
    if 'search_content' in df.columns and df['search_content'].astype(str).str.strip().any():
        return 'search_content'
    for candidate in ['long_description','description','title']:
        if candidate in df.columns:
            return candidate
    return df.columns[0]

def build_tfidf_index(df: pd.DataFrame, field: str):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
    mat = vec.fit_transform(df[field].astype(str).fillna('').values)
    return vec, mat

def query_search_from_index(query: str,
                             df: pd.DataFrame,
                             vectorizer: TfidfVectorizer,
                             tfidf_matrix,
                             text_field: str,
                             top_k: int):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).ravel()
    idxs = sims.argsort()[::-1][:top_k]
    rows = []
    for i in idxs:
        rows.append({
            'clean_text': df.iloc[i].get('search_content', '') or df.iloc[i].get(text_field, ''),
            'product_title': df.iloc[i].get('title',''),
            'url': df.iloc[i].get('product_url','') or df.iloc[i].get('url',''),
            'score': float(sims[i]),
            'sku': df.iloc[i].get('sku','')
        })
    return pd.DataFrame(rows)

def extractive_summary_from_retrieved(retrieved_df, top_k=TOP_K_DEFAULT):
    if retrieved_df is None or retrieved_df.empty:
        return ""
    merged = " ".join(str(x) for x in retrieved_df['clean_text'].astype(str).head(top_k).tolist())
    merged = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', ' ', merged)
    sents = re.split(r'(?<=[.!?])\s+', merged)
    return " ".join(sents[:2]).strip()

def handle_query_with_df(df: pd.DataFrame, query: str, top_k: int = TOP_K_DEFAULT):
    """
    df: dataframe loaded from CSV
    query: user query string
    returns: dict {query, top_results, final_answer, product_links}
    """
    fallback_msg = "Sorry, I couldn't answer that. I can assist you with product, website, business or item-related queries."

    if is_irrelevant(query):
        return {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}

    text_field = choose_text_field(df)
    vectorizer, tfidf_matrix = build_tfidf_index(df, text_field)

    retrieved = query_search_from_index(query=query,
                                        df=df,
                                        vectorizer=vectorizer,
                                        tfidf_matrix=tfidf_matrix,
                                        text_field=text_field,
                                        top_k=top_k)
    if retrieved.empty:
        return {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}

    best_score = float(retrieved['score'].max())
    if best_score < SIMILARITY_THRESHOLD:
        return {
            "query": query,
            "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
            "final_answer": fallback_msg,
            "product_links": []
        }

    final_answer = extractive_summary_from_retrieved(retrieved, top_k=top_k) or fallback_msg
    product_links = [r.get('url','') for _, r in retrieved.head(top_k).iterrows() if r.get('url')]

    out = {
        "query": query,
        "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
        "final_answer": final_answer,
        "product_links": product_links
    }
    return out

def save_result_as_json(result: dict, base_name: str = "query_response"):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = int(time.time())
    path = os.path.join(OUTPUT_DIR, f"{base_name}_{ts}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_jsonify(result), f, indent=2, ensure_ascii=False)
    return path

# ---------- LOAD CSV ONCE ----------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, dtype=str).fillna('')
print(f"Loaded CSV with {len(df)} rows from {CSV_PATH}")

# ---------- INTERACTIVE QUERY ----------
# You can change this to a fixed string instead of input() if you prefer.
user_query = input("Enter your question about the products: ").strip()

result = handle_query_with_df(df, user_query, top_k=TOP_K_DEFAULT)

print("\n--- Final Answer ---")
print(result["final_answer"])
print("\nProduct links:")
for link in result["product_links"]:
    print(" -", link)

# Save to JSON file (optional)
saved_path = save_result_as_json(result, base_name="query_response")
print(f"\nResponse saved to: {saved_path}")

# If you want to inspect the whole JSON here:
import pprint; pprint.pp(result)

Loaded CSV with 92 rows from /kaggle/working/99wholesale_collections_trending-product.csv


Enter your question about the products:  Greek Yogurt & Curd Maker



--- Final Answer ---
Greek Yogurt   Curd Maker   Fine Mesh Strainer with Lid for Homemade Yogurt, Paneer   Cheese  Pack of 1  Product Specifications Material  Food-grade plastic container with fine nylon mesh Design  2-in-1 container with strainer and transparent lid Capacity  1 Litres  approx. Product Specifications Material  Food-grade plastic container with fine nylon mesh Design  2-in-1 container with strainer and transparent lid Capacity  1 Litres  approx.

Product links:
 - https://99wholesale.com/collections/trending-product/products/greek-yogurt-curd-maker-fine-mesh-strainer-with-lid?variant=50876259434805
 - https://99wholesale.com/collections/trending-product/products/mini-waffle-maker-portable-electric-round-waffle-maker-grill-machine-for-individual-pancakes-cookies-eggs-individual-waffles-paninis-hash-browns?variant=51445736374581
 - https://99wholesale.com/collections/trending-product/products/4-pcs-potato-spiral-cutter?variant=50456822645045

Response saved to: /kaggle/w

## Greek Yogurt & Curd Maker - Version 2

### GangsLifeStyle

In [ ]:
# === MULTI-TURN QUERY HANDLER FOR KAGGLE NOTEBOOK ===
# Remembers last product and uses it for follow-up questions like "Price of the bag?"
# Uses `summary` for general answers, and `long_description` when user asks for *more* details.

import os
import re
import json
import time

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional NLTK for sentence splitting
import nltk
from nltk.tokenize import sent_tokenize

# minimal nltk assets required (download if missing)
for r in ['punkt']:
    try:
        nltk.data.find(r)
    except Exception:
        try:
            nltk.download(r, quiet=True)
        except Exception:
            pass

# ---------- CONFIG ----------
CSV_PATH = "/kaggle/working/gangslifestyle.csv"   # <- change if needed
SIMILARITY_THRESHOLD = 0.30
TOP_K_DEFAULT = 3
OUTPUT_DIR = "/kaggle/working"

# ---------- HELPERS ----------
IRRELEVANT_KEYWORDS = [
    "gym","dumbbell","exercise","workout","recipe","food","cooking",
    "weather","news","politics","relationship","doctor","medicine",
    "math","code","python","java","cpp"
]
_irrelevant_regex = re.compile(r'\b(' + r'|'.join(re.escape(w) for w in IRRELEVANT_KEYWORDS) + r')\b', flags=re.I)

def is_irrelevant(query: str) -> bool:
    if not query or str(query).strip() == "":
        return True
    return bool(_irrelevant_regex.search(query))

def safe_jsonify(obj):
    """Convert numpy / pandas types to JSON-serializable types."""
    if isinstance(obj, dict):
        return {safe_jsonify(k): safe_jsonify(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [safe_jsonify(x) for x in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return obj

def choose_text_field(df: pd.DataFrame) -> str:
    """Pick the best field to index for retrieval (prefer indexed_text_lemma, then search_content, else fallback)."""
    if 'indexed_text_lemma' in df.columns and df['indexed_text_lemma'].astype(str).str.strip().any():
        return 'indexed_text_lemma'
    if 'search_content' in df.columns and df['search_content'].astype(str).str.strip().any():
        return 'search_content'
    for candidate in ['long_description','description','title']:
        if candidate in df.columns:
            return candidate
    return df.columns[0]

def build_tfidf_index(df: pd.DataFrame, field: str):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
    mat = vec.fit_transform(df[field].astype(str).fillna('').values)
    return vec, mat

def query_search_from_index(query: str,
                             df: pd.DataFrame,
                             vectorizer: TfidfVectorizer,
                             tfidf_matrix,
                             text_field: str,
                             top_k: int):
    """
    Returns a DataFrame with:
      clean_text, product_title, url, score, sku,
      price_current, price_original, discount_percent, stock_status,
      summary, long_description
    """
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).ravel()
    idxs = sims.argsort()[::-1][:top_k]
    rows = []
    for i in idxs:
        row = df.iloc[i]
        rows.append({
            'clean_text': row.get('search_content', '') or row.get(text_field, ''),
            'product_title': row.get('title',''),
            'url': row.get('product_url','') or row.get('url',''),
            'score': float(sims[i]),
            'sku': row.get('sku',''),
            'price_current': row.get('price_current',''),
            'price_original': row.get('price_original',''),
            'discount_percent': row.get('discount_percent',''),
            'stock_status': row.get('stock_status',''),
            'summary': row.get('summary',''),
            'long_description': row.get('long_description','')
        })
    return pd.DataFrame(rows)

def extractive_summary_from_retrieved(retrieved_df, top_k=TOP_K_DEFAULT):
    if retrieved_df is None or retrieved_df.empty:
        return ""
    merged = " ".join(str(x) for x in retrieved_df['clean_text'].astype(str).head(top_k).tolist())
    merged = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', ' ', merged)
    # split into sentences
    sents = re.split(r'(?<=[.!?])\s+', merged)
    return " ".join(sents[:2]).strip()

# ---------- FOLLOW-UP DETECTION ----------
def looks_like_followup(query: str, state: dict) -> bool:
    """
    Decide if this question should use the last product context.
    We want single-word follow-ups like "Price", "Discount", "Stock", "Status"
    to automatically refer to the last product.
    """
    q = query.strip().lower()
    if not state.get("last_top_product_title"):
        return False

    # obvious conversational starters
    if q.startswith("then ") or q.startswith("what about") or q.startswith("and "):
        return True

    # generic follow-up keywords (NOW includes 'discount' and 'offer')
    follow_keywords = [
        "price", "cost", "rate",
        "discount", "offer",
        "color", "colour", "size",
        "details", "more about", "explain",
        "stock", "availability", "status"
    ]
    if any(kw in q for kw in follow_keywords) and len(q.split()) <= 7:
        return True

    # references to previous / above product
    if ("above" in q or "previous" in q or "earlier" in q) and any(
        w in q for w in ["product", "item", "one"]
    ):
        return True

    # vague references, still treat as follow-up if short
    if any(w in q for w in ["this", "that", "it", "above", "previous", "earlier"]) and len(q.split()) <= 10:
        return True

    # extra safety: if query is VERY short (1–2 words) and looks like a price/stock/discount intent, use follow-up
    if len(q.split()) <= 2 and detect_question_intent(query) in ["price", "discount", "stock", "orig_price"]:
        return True

    return False


# ---------- INTENT DETECTION ----------
def detect_question_intent(query: str) -> str:
    """
    Returns one of: 'discount', 'orig_price', 'price', 'stock', 'detail', 'general'
    """
    q = query.lower()
    # discount / offer
    if any(kw in q for kw in ["discount", "offer", "% off", "off "]):
        return "discount"
    # original price / mrp
    if any(kw in q for kw in ["original price", "mrp", "actual price", "before discount"]):
        return "orig_price"
    # current price / price / cost
    if any(kw in q for kw in ["price", "cost", "rate"]):
        return "price"
    # stock / availability / status
    if any(kw in q for kw in ["stock", "in stock", "out of stock", "available", "availability", "status"]):
        return "stock"
    # more details / explanation (NOTE: we only trigger on *more* / explain style, not just the word "details")
    detail_phrases = [
        "more detail", "more details", "more in detail",
        "tell me more", "explain", "explanation",
        "full description", "describe", "more info", "more about this"
    ]
    if any(p in q for p in detail_phrases):
        return "detail"
    return "general"

# ---------- MAIN HANDLER WITH CONTEXT ----------
def handle_query_with_context(df: pd.DataFrame,
                              query: str,
                              vectorizer: TfidfVectorizer,
                              tfidf_matrix,
                              text_field: str,
                              state: dict,
                              top_k: int = TOP_K_DEFAULT):
    """
    df: dataframe loaded from CSV
    query: user question
    vectorizer, tfidf_matrix, text_field: pre-built retrieval objects
    state: dict with conversation memory (mutated in-place)
    """
    fallback_msg = "Sorry, I couldn't answer that. I can assist you with product, website, business, or item-related queries."

    if is_irrelevant(query):
        result = {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}
        state["last_query"] = query
        state["last_result"] = None
        state["last_top_product_title"] = None
        return result, state

    # Decide whether to use previous context
    use_followup = looks_like_followup(query, state)

    if use_followup:
        base_title = state.get("last_top_product_title", "")
        expanded_query = f"{query.strip()} for product: {base_title}"
        search_query = expanded_query
    else:
        search_query = query

    # Retrieval
    retrieved = query_search_from_index(query=search_query,
                                        df=df,
                                        vectorizer=vectorizer,
                                        tfidf_matrix=tfidf_matrix,
                                        text_field=text_field,
                                        top_k=top_k)
    if retrieved.empty:
        result = {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}
        state["last_query"] = query
        state["last_result"] = retrieved
        state["last_top_product_title"] = None
        return result, state

    best_score = float(retrieved['score'].max())
    if best_score < SIMILARITY_THRESHOLD:
        result = {
            "query": query,
            "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
            "final_answer": fallback_msg,
            "product_links": []
        }
        state["last_query"] = query
        state["last_result"] = retrieved
        state["last_top_product_title"] = None
        return result, state

    # ---------- SPECIAL ANSWERS ----------
    intent = detect_question_intent(query)
    top = retrieved.iloc[0]
    title = (top.get("product_title") or "").strip()

    price_current = str(top.get("price_current") or "").strip()
    price_original = str(top.get("price_original") or "").strip()
    discount_percent = str(top.get("discount_percent") or "").strip()
    stock_status = str(top.get("stock_status") or "").strip()
    summary_txt = str(top.get("summary") or "").strip()
    long_desc = str(top.get("long_description") or "").strip()

    final_answer = None

    # 1) Price-type questions
    if intent == "price":
        if price_current and price_current.lower() != "nan":
            final_answer = f"The current price of {title} is {price_current}."
            if discount_percent and discount_percent.lower() != "nan":
                final_answer += f" It currently has a discount of {discount_percent}%."
        elif price_original and price_original.lower() != "nan":
            final_answer = f"The price information of {title} is not fully available, but the original price is {price_original}."
    elif intent == "orig_price":
        if price_original and price_original.lower() != "nan":
            final_answer = f"The original price (before discount) of {title} is {price_original}."
        elif price_current and price_current.lower() != "nan":
            final_answer = f"The original price is not available, but the current price of {title} is {price_current}."
    elif intent == "discount":
        if discount_percent and discount_percent.lower() != "nan":
            final_answer = f"{title} currently has a discount of {discount_percent}%."
        elif price_original and price_current and price_original.lower() != "nan" and price_current.lower() != "nan":
            try:
                po = float(price_original)
                pc = float(price_current)
                if po > 0:
                    disc = round((po - pc) / po * 100, 1)
                    final_answer = f"{title} has an approximate discount of {disc}%."
            except Exception:
                pass
    elif intent == "stock":
        if stock_status and stock_status.lower() != "nan":
            final_answer = f"{title} is currently {stock_status}."
        else:
            final_answer = f"The stock status of {title} is not clearly available."
    # 2) Detail questions → use long_description
    elif intent == "detail":
        if long_desc and long_desc.lower() != "nan":
            final_answer = f"Here are more details about {title}: {long_desc}"
        elif summary_txt:
            final_answer = f"Here is a summary of {title}: {summary_txt}"
    # 3) General questions → use summary first
    if intent == "general" and not final_answer:
        if summary_txt and summary_txt.lower() != "nan":
            final_answer = f"{title}: {summary_txt}"
        else:
            # fallback to extractive summary
            final_answer = extractive_summary_from_retrieved(retrieved, top_k=top_k) or fallback_msg

    # Safety: if none of the above set final_answer, still fallback
    if not final_answer:
        if summary_txt:
            final_answer = f"{title}: {summary_txt}"
        else:
            final_answer = extractive_summary_from_retrieved(retrieved, top_k=top_k) or fallback_msg

    product_links = [r.get('url','') for _, r in retrieved.head(top_k).iterrows() if r.get('url')]

    result = {
        "query": query,
        "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
        "final_answer": final_answer,
        "product_links": product_links
    }

    # Update conversation state
    state["last_query"] = query
    state["last_result"] = retrieved
    state["last_top_product_title"] = top.get("product_title") or ""

    return result, state

def save_result_as_json(result: dict, base_name: str = "query_response"):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = int(time.time())
    path = os.path.join(OUTPUT_DIR, f"{base_name}_{ts}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_jsonify(result), f, indent=2, ensure_ascii=False)
    return path

# ---------- LOAD CSV + BUILD INDEX ONCE ----------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, dtype=str).fillna('')
print(f"Loaded CSV with {len(df)} rows from {CSV_PATH}")

text_field = choose_text_field(df)
print("Indexing text field:", text_field)
vectorizer, tfidf_matrix = build_tfidf_index(df, text_field)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# ---------- INTERACTIVE LOOP WITH CONTEXT ----------
conversation_state = {
    "last_query": None,
    "last_result": None,
    "last_top_product_title": None
}

print("\nType your questions. Type 'exit' or 'quit' to stop.\n")

while True:
    user_query = input("You: ").strip()
    if user_query.lower() in ["exit", "quit"]:
        print("Exiting.")
        break

    result, conversation_state = handle_query_with_context(
        df=df,
        query=user_query,
        vectorizer=vectorizer,
        tfidf_matrix=tfidf_matrix,
        text_field=text_field,
        state=conversation_state,
        top_k=TOP_K_DEFAULT
    )

    print("\nBot:", result["final_answer"])
    if result["product_links"]:
        print("Links:")
        for link in result["product_links"]:
            print(" -", link)

    save_path = save_result_as_json(result, base_name="query_response")
    print(f"(Saved response JSON to: {save_path})\n")


### 99Wholesale

In [13]:
# === MULTI-TURN QUERY HANDLER FOR KAGGLE NOTEBOOK ===
# Remembers last product and uses it for follow-up questions like "Price of the bag?"
# Uses `summary` for general answers, and `long_description` when user asks for *more* details.

import os
import re
import json
import time

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional NLTK for sentence splitting
import nltk
from nltk.tokenize import sent_tokenize

# minimal nltk assets required (download if missing)
for r in ['punkt']:
    try:
        nltk.data.find(r)
    except Exception:
        try:
            nltk.download(r, quiet=True)
        except Exception:
            pass

# ---------- CONFIG ----------
CSV_PATH = "/kaggle/working/99wholesale_collections_trending-product.csv"   # <- change if needed
SIMILARITY_THRESHOLD = 0.30
TOP_K_DEFAULT = 3
OUTPUT_DIR = "/kaggle/working"

# ---------- HELPERS ----------
IRRELEVANT_KEYWORDS = [
    "gym","dumbbell","exercise","workout","recipe","food","cooking",
    "weather","news","politics","relationship","doctor","medicine",
    "math","code","python","java","cpp"
]
_irrelevant_regex = re.compile(r'\b(' + r'|'.join(re.escape(w) for w in IRRELEVANT_KEYWORDS) + r')\b', flags=re.I)

def is_irrelevant(query: str) -> bool:
    if not query or str(query).strip() == "":
        return True
    return bool(_irrelevant_regex.search(query))

def safe_jsonify(obj):
    """Convert numpy / pandas types to JSON-serializable types."""
    if isinstance(obj, dict):
        return {safe_jsonify(k): safe_jsonify(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [safe_jsonify(x) for x in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if pd.isna(obj):
        return None
    return obj

def choose_text_field(df: pd.DataFrame) -> str:
    """Pick the best field to index for retrieval (prefer indexed_text_lemma, then search_content, else fallback)."""
    if 'indexed_text_lemma' in df.columns and df['indexed_text_lemma'].astype(str).str.strip().any():
        return 'indexed_text_lemma'
    if 'search_content' in df.columns and df['search_content'].astype(str).str.strip().any():
        return 'search_content'
    for candidate in ['long_description','description','title']:
        if candidate in df.columns:
            return candidate
    return df.columns[0]

def build_tfidf_index(df: pd.DataFrame, field: str):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
    mat = vec.fit_transform(df[field].astype(str).fillna('').values)
    return vec, mat

def query_search_from_index(query: str,
                             df: pd.DataFrame,
                             vectorizer: TfidfVectorizer,
                             tfidf_matrix,
                             text_field: str,
                             top_k: int):
    """
    Returns a DataFrame with:
      clean_text, product_title, url, score, sku,
      price_current, price_original, discount_percent, stock_status,
      summary, long_description
    """
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).ravel()
    idxs = sims.argsort()[::-1][:top_k]
    rows = []
    for i in idxs:
        row = df.iloc[i]
        rows.append({
            'clean_text': row.get('search_content', '') or row.get(text_field, ''),
            'product_title': row.get('title',''),
            'url': row.get('product_url','') or row.get('url',''),
            'score': float(sims[i]),
            'sku': row.get('sku',''),
            'price_current': row.get('price_current',''),
            'price_original': row.get('price_original',''),
            'discount_percent': row.get('discount_percent',''),
            'stock_status': row.get('stock_status',''),
            'summary': row.get('summary',''),
            'long_description': row.get('long_description','')
        })
    return pd.DataFrame(rows)

def extractive_summary_from_retrieved(retrieved_df, top_k=TOP_K_DEFAULT):
    if retrieved_df is None or retrieved_df.empty:
        return ""
    merged = " ".join(str(x) for x in retrieved_df['clean_text'].astype(str).head(top_k).tolist())
    merged = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', ' ', merged)
    # split into sentences
    sents = re.split(r'(?<=[.!?])\s+', merged)
    return " ".join(sents[:2]).strip()

# ---------- FOLLOW-UP DETECTION ----------
def looks_like_followup(query: str, state: dict) -> bool:
    """
    Decide if this question should use the last product context.
    We want single-word follow-ups like "Price", "Discount", "Stock", "Status"
    to automatically refer to the last product.
    """
    q = query.strip().lower()
    if not state.get("last_top_product_title"):
        return False

    # obvious conversational starters
    if q.startswith("then ") or q.startswith("what about") or q.startswith("and "):
        return True

    # generic follow-up keywords (NOW includes 'discount' and 'offer')
    follow_keywords = [
        "price", "cost", "rate",
        "discount", "offer",
        "color", "colour", "size",
        "details", "more about", "explain",
        "stock", "availability", "status"
    ]
    if any(kw in q for kw in follow_keywords) and len(q.split()) <= 7:
        return True

    # references to previous / above product
    if ("above" in q or "previous" in q or "earlier" in q) and any(
        w in q for w in ["product", "item", "one"]
    ):
        return True

    # vague references, still treat as follow-up if short
    if any(w in q for w in ["this", "that", "it", "above", "previous", "earlier"]) and len(q.split()) <= 10:
        return True

    # extra safety: if query is VERY short (1–2 words) and looks like a price/stock/discount intent, use follow-up
    if len(q.split()) <= 2 and detect_question_intent(query) in ["price", "discount", "stock", "orig_price"]:
        return True

    return False


# ---------- INTENT DETECTION ----------
def detect_question_intent(query: str) -> str:
    """
    Returns one of: 'discount', 'orig_price', 'price', 'stock', 'detail', 'general'
    """
    q = query.lower()
    # discount / offer
    if any(kw in q for kw in ["discount", "offer", "% off", "off "]):
        return "discount"
    # original price / mrp
    if any(kw in q for kw in ["original price", "mrp", "actual price", "before discount"]):
        return "orig_price"
    # current price / price / cost
    if any(kw in q for kw in ["price", "cost", "rate"]):
        return "price"
    # stock / availability / status
    if any(kw in q for kw in ["stock", "in stock", "out of stock", "available", "availability", "status"]):
        return "stock"
    # more details / explanation (NOTE: we only trigger on *more* / explain style, not just the word "details")
    detail_phrases = [
        "more detail", "more details", "more in detail",
        "tell me more", "explain", "explanation",
        "full description", "describe", "more info", "more about this"
    ]
    if any(p in q for p in detail_phrases):
        return "detail"
    return "general"

# ---------- MAIN HANDLER WITH CONTEXT ----------
def handle_query_with_context(df: pd.DataFrame,
                              query: str,
                              vectorizer: TfidfVectorizer,
                              tfidf_matrix,
                              text_field: str,
                              state: dict,
                              top_k: int = TOP_K_DEFAULT):
    """
    df: dataframe loaded from CSV
    query: user question
    vectorizer, tfidf_matrix, text_field: pre-built retrieval objects
    state: dict with conversation memory (mutated in-place)
    """
    fallback_msg = "Sorry, I couldn't answer that. I can assist you with product, website, business, or item-related queries."

    if is_irrelevant(query):
        result = {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}
        state["last_query"] = query
        state["last_result"] = None
        state["last_top_product_title"] = None
        return result, state

    # Decide whether to use previous context
    use_followup = looks_like_followup(query, state)

    if use_followup:
        base_title = state.get("last_top_product_title", "")
        expanded_query = f"{query.strip()} for product: {base_title}"
        search_query = expanded_query
    else:
        search_query = query

    # Retrieval
    retrieved = query_search_from_index(query=search_query,
                                        df=df,
                                        vectorizer=vectorizer,
                                        tfidf_matrix=tfidf_matrix,
                                        text_field=text_field,
                                        top_k=top_k)
    if retrieved.empty:
        result = {"query": query, "top_results": [], "final_answer": fallback_msg, "product_links": []}
        state["last_query"] = query
        state["last_result"] = retrieved
        state["last_top_product_title"] = None
        return result, state

    best_score = float(retrieved['score'].max())
    if best_score < SIMILARITY_THRESHOLD:
        result = {
            "query": query,
            "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
            "final_answer": fallback_msg,
            "product_links": []
        }
        state["last_query"] = query
        state["last_result"] = retrieved
        state["last_top_product_title"] = None
        return result, state

    # ---------- SPECIAL ANSWERS ----------
    intent = detect_question_intent(query)
    top = retrieved.iloc[0]
    title = (top.get("product_title") or "").strip()

    price_current = str(top.get("price_current") or "").strip()
    price_original = str(top.get("price_original") or "").strip()
    discount_percent = str(top.get("discount_percent") or "").strip()
    stock_status = str(top.get("stock_status") or "").strip()
    summary_txt = str(top.get("summary") or "").strip()
    long_desc = str(top.get("long_description") or "").strip()

    final_answer = None

    # 1) Price-type questions
    if intent == "price":
        if price_current and price_current.lower() != "nan":
            final_answer = f"The current price of {title} is {price_current}."
            if discount_percent and discount_percent.lower() != "nan":
                final_answer += f" It currently has a discount of {discount_percent}%."
        elif price_original and price_original.lower() != "nan":
            final_answer = f"The price information of {title} is not fully available, but the original price is {price_original}."
    elif intent == "orig_price":
        if price_original and price_original.lower() != "nan":
            final_answer = f"The original price (before discount) of {title} is {price_original}."
        elif price_current and price_current.lower() != "nan":
            final_answer = f"The original price is not available, but the current price of {title} is {price_current}."
    elif intent == "discount":
        if discount_percent and discount_percent.lower() != "nan":
            final_answer = f"{title} currently has a discount of {discount_percent}%."
        elif price_original and price_current and price_original.lower() != "nan" and price_current.lower() != "nan":
            try:
                po = float(price_original)
                pc = float(price_current)
                if po > 0:
                    disc = round((po - pc) / po * 100, 1)
                    final_answer = f"{title} has an approximate discount of {disc}%."
            except Exception:
                pass
    elif intent == "stock":
        if stock_status and stock_status.lower() != "nan":
            final_answer = f"{title} is currently {stock_status}."
        else:
            final_answer = f"The stock status of {title} is not clearly available."
    # 2) Detail questions → use long_description
    elif intent == "detail":
        if long_desc and long_desc.lower() != "nan":
            final_answer = f"Here are more details about {title}: {long_desc}"
        elif summary_txt:
            final_answer = f"Here is a summary of {title}: {summary_txt}"
    # 3) General questions → use summary first
    if intent == "general" and not final_answer:
        if summary_txt and summary_txt.lower() != "nan":
            final_answer = f"{title}: {summary_txt}"
        else:
            # fallback to extractive summary
            final_answer = extractive_summary_from_retrieved(retrieved, top_k=top_k) or fallback_msg

    # Safety: if none of the above set final_answer, still fallback
    if not final_answer:
        if summary_txt:
            final_answer = f"{title}: {summary_txt}"
        else:
            final_answer = extractive_summary_from_retrieved(retrieved, top_k=top_k) or fallback_msg

    product_links = [r.get('url','') for _, r in retrieved.head(top_k).iterrows() if r.get('url')]

    result = {
        "query": query,
        "top_results": safe_jsonify(retrieved.to_dict(orient='records')),
        "final_answer": final_answer,
        "product_links": product_links
    }

    # Update conversation state
    state["last_query"] = query
    state["last_result"] = retrieved
    state["last_top_product_title"] = top.get("product_title") or ""

    return result, state

def save_result_as_json(result: dict, base_name: str = "query_response"):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = int(time.time())
    path = os.path.join(OUTPUT_DIR, f"{base_name}_{ts}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_jsonify(result), f, indent=2, ensure_ascii=False)
    return path

# ---------- LOAD CSV + BUILD INDEX ONCE ----------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, dtype=str).fillna('')
print(f"Loaded CSV with {len(df)} rows from {CSV_PATH}")

text_field = choose_text_field(df)
print("Indexing text field:", text_field)
vectorizer, tfidf_matrix = build_tfidf_index(df, text_field)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# ---------- INTERACTIVE LOOP WITH CONTEXT ----------
conversation_state = {
    "last_query": None,
    "last_result": None,
    "last_top_product_title": None
}

print("\nType your questions. Type 'exit' or 'quit' to stop.\n")

while True:
    user_query = input("You: ").strip()
    if user_query.lower() in ["exit", "quit"]:
        print("Exiting.")
        break

    result, conversation_state = handle_query_with_context(
        df=df,
        query=user_query,
        vectorizer=vectorizer,
        tfidf_matrix=tfidf_matrix,
        text_field=text_field,
        state=conversation_state,
        top_k=TOP_K_DEFAULT
    )

    print("\nBot:", result["final_answer"])
    if result["product_links"]:
        print("Links:")
        for link in result["product_links"]:
            print(" -", link)

    save_path = save_result_as_json(result, base_name="query_response")
    print(f"(Saved response JSON to: {save_path})\n")


Loaded CSV with 92 rows from /kaggle/working/99wholesale_collections_trending-product.csv
Indexing text field: indexed_text_lemma
TF-IDF matrix shape: (92, 15818)

Type your questions. Type 'exit' or 'quit' to stop.



You:  Details about Foldable Storage Organizer Basket with Lid



Bot: Foldable Storage Organizer Basket with Lid – Transparent Fabric Closet & Wardrobe Storage Box for Clothes (Pack of 1): Foldable Design: Maximize your space when not in use!
Links:
 - https://99wholesale.com/collections/trending-product/products/foldable-storage-organizer-basket-with-lid?variant=50457133056309
 - https://99wholesale.com/collections/trending-product/products/greek-yogurt-curd-maker-fine-mesh-strainer-with-lid?variant=50876259434805
 - https://99wholesale.com/collections/trending-product/products/360-rotating-adjustable-cosmetic-storage?variant=50456774017333
(Saved response JSON to: /kaggle/working/query_response_1764001418.json)



You:  Price



Bot: The current price of Foldable Storage Organizer Basket with Lid – Transparent Fabric Closet & Wardrobe Storage Box for Clothes (Pack of 1) is 299.0. It currently has a discount of 0.0%.
Links:
 - https://99wholesale.com/collections/trending-product/products/foldable-storage-organizer-basket-with-lid?variant=50457133056309
 - https://99wholesale.com/collections/trending-product/products/clear-foldable-shoe-bag?variant=50907729920309
 - https://99wholesale.com/collections/trending-product/products/4-pcs-universal-wheel-rack?variant=50456825004341
(Saved response JSON to: /kaggle/working/query_response_1764001421.json)



...


Bot: Premium Car Sunshade - Ultimate UV Protection & Heat Blocker for Windshield (Pack of 1): Product Specifications Material: High-quality reflective fabric with thermal insulation coating Design: Foldable, lightweight, and easy-to-install windshield cover Function: Blocks harmful UV rays, reduces heat buildup, protects interiors Compatibility: Suitable for most cars, SUVs, and small trucks Approximate Dimensions: Length – 150 cm | Width – 70 cm Weight: Lightweight and portable for convenient use and storage User Benefits Superior UV Protection: Shields your car’s interior from harmful sun rays that fade upholstery and damage dashboards.
Links:
 - https://99wholesale.com/collections/trending-product/products/1pc-premium-car-sunshade-ultimate-uv-protection-heat-blocker?variant=50842274726197
 - https://99wholesale.com/collections/trending-product/products/mini-portable-outdoor-gas-stove?variant=50457692471605
 - https://99wholesale.com/collections/trending-product/products/3-in-1-micr

You:  Price



Bot: The current price of Premium Car Sunshade - Ultimate UV Protection & Heat Blocker for Windshield (Pack of 1) is 299.0. It currently has a discount of 0.0%.
Links:
 - https://99wholesale.com/collections/trending-product/products/1pc-premium-car-sunshade-ultimate-uv-protection-heat-blocker?variant=50842274726197
 - https://99wholesale.com/collections/trending-product/products/mini-portable-outdoor-gas-stove?variant=50457692471605
 - https://99wholesale.com/collections/trending-product/products/3-in-1-microfiber-duster-premium-quality?variant=50456752849205
(Saved response JSON to: /kaggle/working/query_response_1764001477.json)



You:  exit 


Exiting.
